In [10]:
## ===============================
## Import Libraries
## ===============================

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader


## ===============================
## Image Transformations
## ===============================

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])


## ===============================
## Load Dataset
## ===============================

train_dataset = ImageFolder(
    "dataset/train",
    transform=transform
)

print("Classes :", train_dataset.classes)
print("Number of Images :", len(train_dataset))


## ===============================
## DataLoader
## ===============================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)


## ===============================
## CNN Model
## ===============================

class CNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Conv2d(3,16,kernel_size=3,padding=1),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(16,32,kernel_size=3,padding=1),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(32*32*32,128),

            nn.ReLU(),

            nn.Linear(128,2)

        )

    def forward(self,x):

        debug = False

        for layer in self.network:

            x = layer(x)

            if debug:
                print(layer)
                print(x.shape)
                print("----------------")

        return x


## ===============================
## Create Model
## ===============================

model = CNN()

model.train()


## ===============================
## Loss Function
## ===============================

criterion = nn.CrossEntropyLoss()


## ===============================
## Optimizer
## ===============================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


## ===============================
## Training
## ===============================

epochs = 10

for epoch in range(epochs):

    running_loss = 0

    for images, labels in train_loader:

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss : {running_loss/len(train_loader):.4f}"
    )


## ===============================
## Save Model
## ===============================

torch.save(
    model.state_dict(),
    "saved_model.pth"
)

print("Model Saved Successfully!")

Classes : ['cats', 'dogs']
Number of Images : 8005
Epoch [1/10] Loss : 0.6556
Epoch [2/10] Loss : 0.5383
Epoch [3/10] Loss : 0.4534
Epoch [4/10] Loss : 0.3616
Epoch [5/10] Loss : 0.2494
Epoch [6/10] Loss : 0.1272
Epoch [7/10] Loss : 0.0645
Epoch [8/10] Loss : 0.0266
Epoch [9/10] Loss : 0.0127
Epoch [10/10] Loss : 0.0047
Model Saved Successfully!


In [12]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image


# ==========================
# CNN Model
# ==========================

class CNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Conv2d(3,16,kernel_size=3,padding=1),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(16,32,kernel_size=3,padding=1),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(32*32*32,128),

            nn.ReLU(),

            nn.Linear(128,2)

        )

    def forward(self,x):

        return self.network(x)


# ==========================
# Load Model
# ==========================

model = CNN()

model.load_state_dict(torch.load("saved_model.pth"))

model.eval()


# ==========================
# Image Transform
# ==========================

transform = transforms.Compose([

    transforms.Resize((128,128)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )

])


# ==========================
# Load Image
# ==========================

image = Image.open("manja-vitolic-gKXKBY-C-Dk-unsplash.jpg")

image = image.convert("RGB")

image = transform(image)

image = image.unsqueeze(0)


# ==========================
# Prediction
# ==========================

with torch.no_grad():

    output = model(image)

    predicted = torch.argmax(output, dim=1)


# ==========================
# Class Names
# ==========================

classes = ["Cat", "Dog"]

print("Prediction :", classes[predicted.item()])

Prediction : Cat
